## Configuration

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
# Path configuration
# path_file = '/content/drive/MyDrive/tesisugm/'  # google colab
path_file = ''  # local
file_path = os.path.join(path_file, "", "utils_ml.py")

In [ ]:
%%writefile $file_path


# Nama File: utils_ml.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from scipy.sparse import issparse

from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os

FEATURES_VERSION = "utils_ml_0.2.6"

# ==============================================================
# 0. DYNAMIC DIRECTORY SETTINGS
# ==============================================================
def save_to_dir(dir="images", name="image"):
    os.makedirs(dir, exist_ok=True)
    path = os.path.join(dir, f"{name}.png")
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f"🖼️ Saved: {path}")
    plt.show()
    plt.close()

# ==============================================================
# 1. GLOBAL THEME & STYLING
# ==============================================================
PALETTE    = ["#2D4A7A", "#4A90C4", "#6DB8A8", "#C4956A"]  # Jordan, Lebanon, Palestine, Syria
BG_COLOR   = "#F8F9FB"
CARD_COLOR = "#FFFFFF"
TEXT_DARK  = "#1A2640"
TEXT_MID   = "#4A5568"
TEXT_LIGHT = "#8A97AA"
GRID_COLOR = "#E8ECF1"

def set_global_style():
    """Mengatur tema visual matplotlib secara global untuk semua notebook."""
    plt.rcParams.update({
        "figure.facecolor"  : BG_COLOR,
        "axes.facecolor"    : CARD_COLOR,
        "axes.edgecolor"    : GRID_COLOR,
        "axes.linewidth"    : 0.8,
        "axes.grid"         : True,
        "grid.color"        : GRID_COLOR,
        "grid.linewidth"    : 0.6,
        "grid.alpha"        : 0.8,
        "text.color"        : TEXT_DARK,
        "axes.labelcolor"   : TEXT_MID,
        "xtick.color"       : TEXT_MID,
        "ytick.color"       : TEXT_MID,
        "xtick.labelsize"   : 10,
        "ytick.labelsize"   : 10,
        "axes.labelsize"    : 11,
        "axes.titlesize"    : 13,
        "axes.titleweight"  : "bold",
        "axes.titlepad"     : 12,
        "font.family"       : "sans-serif",
        "axes.spines.top"   : False,
        "axes.spines.right" : False,
    })

# Panggil langsung saat modul di-import
set_global_style()

# ==============================================================
# 2. VISUALIZATION & INFERENCE HELPERS
# ==============================================================
def plot_before_after_smote(y_train, y_resampled, label_names, title_suffix="", delim="", dir="images", name="image"):
    """Visualisasi Bar Chart Before vs After Oversampling."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5)) # Facecolor otomatis ngikutin global
    
    for ax, (y_data, title) in zip(axes, [(y_train, "Before SMOTE"), (y_resampled, f"After {title_suffix}")]):
        counts_plot = [np.sum(y_data == i) for i in range(len(label_names))]
        
        # Gambar bar chart
        bars = ax.bar(label_names, counts_plot, color=PALETTE[:len(label_names)], 
                      edgecolor=BG_COLOR, linewidth=1.5)
        
        # Tambahkan teks angka di atas batang
        for bar, val in zip(bars, counts_plot):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(counts_plot)*0.01,
                    f"{val:,}", ha='center', va='bottom',
                    fontsize=9, fontweight='bold', color=TEXT_MID)
            
        ax.set_title(title)
        ax.set_ylabel("Count")
        ax.set_axisbelow(True) # Pastikan grid ada di belakang bar

    fig.suptitle(f"Class Distribution{delim}{title_suffix}", fontsize=15, fontweight='bold', color=TEXT_DARK)
    fig.tight_layout()
    save_to_dir(dir, name)

def predict_custom(text, model, vectorizer, label_encoder):
    """Testing model dengan kalimat kustom."""
    feat = vectorizer.transform([text])
    proba = model.predict_proba(feat)[0]
    pred_idx = np.argmax(proba)
    
    print("=" * 45)
    print(f"Input   : {text}")
    print("-" * 45)
    print(f"Prediksi: {label_encoder.classes_[pred_idx]}  ({proba[pred_idx]*100:.1f}%)")
    print("-" * 45)
    print("Probabilitas semua kelas:")
    
    for idx in np.argsort(proba)[::-1]:
        bar = "█" * int(proba[idx] * 30)
        print(f"  {label_encoder.classes_[idx]:12s}: {proba[idx]*100:5.1f}%  {bar}")
    print("=" * 45)
    
    return label_encoder.classes_[pred_idx], proba

# ==============================================================
# 3. ANALYSIS MODULES (Quality, Intrusion, Boundary)
# ==============================================================
def synthetic_quality_analysis(X_orig, X_synth, y_orig, y_synth, label_names, method_name):
    """Bandingkan distribusi jarak NN (Asli vs Sintetis)."""
    results = {}
    classes = np.unique(y_orig)
    for c in classes:
        cls_name  = label_names[c]
        idx_orig  = np.where(y_orig  == c)[0]
        idx_synth = np.where(y_synth == c)[0]
        if len(idx_synth) == 0: continue

        X_c_orig  = X_orig[idx_orig]
        X_c_synth = X_synth[idx_synth]
        if issparse(X_c_orig):
            X_c_orig  = X_c_orig.toarray()
            X_c_synth = X_c_synth.toarray()

        k   = min(5, len(idx_orig) - 1)
        nn  = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
        nn.fit(X_c_orig)

        dist_orig, _  = nn.kneighbors(X_c_orig)
        dist_orig     = dist_orig[:, 1:].mean(axis=1)

        n_syn = min(len(idx_synth), 2000)
        rng   = np.random.default_rng(42)
        s_idx = rng.choice(len(idx_synth), n_syn, replace=False)
        dist_synth, _ = nn.kneighbors(X_c_synth[s_idx])
        dist_synth    = dist_synth[:, 0:].mean(axis=1)

        results[cls_name] = {
            "orig_mean"  : float(np.mean(dist_orig)),
            "orig_std"   : float(np.std(dist_orig)),
            "synth_mean" : float(np.mean(dist_synth)),
            "synth_std"  : float(np.std(dist_synth)),
            "ratio"      : float(np.mean(dist_synth) / (np.mean(dist_orig) + 1e-9)),
        }
    return results

def inter_class_intrusion(X_os, y_os, y_orig, label_names, method_name):
    """Hitung intrusi sampel sintetis ke kelas lain."""
    n_orig   = len(y_orig)
    y_synth  = y_os[n_orig:]
    X_synth  = X_os[n_orig:]
    classes  = np.unique(y_orig)

    X_arr = X_os.toarray() if issparse(X_os) else np.asarray(X_os)
    Xn = normalize(X_arr, norm="l2")

    centroids = {}
    for c in classes:
        idx = np.where(y_orig == c)[0]
        mu  = Xn[idx].mean(axis=0)
        centroids[c] = mu / (np.linalg.norm(mu) + 1e-9)

    X_synth_arr = X_synth.toarray() if issparse(X_synth) else np.asarray(X_synth)
    Xsn = normalize(X_synth_arr, norm="l2")

    intrusion_per_class = {}
    for c in classes:
        idx_c = np.where(y_synth == c)[0]
        if len(idx_c) == 0: continue
        Xsc = Xsn[idx_c]

        sims = np.array([Xsc @ centroids[cc] for cc in classes]).T
        pred = np.argmax(sims, axis=1)

        correct  = np.sum(pred == list(classes).index(c))
        intruded = len(idx_c) - correct
        intrusion_per_class[label_names[c]] = {
            "n_synth"       : len(idx_c),
            "n_intruded"    : int(intruded),
            "intrusion_rate": round(intruded / len(idx_c), 4),
        }
    return intrusion_per_class

def decision_boundary_analysis(model_before, model_after, Xte_sel, y_test, label_names):
    """Bandingkan Decision Boundary 2 Model (Original vs SMOTE)"""
    proba_before = model_before.predict_proba(Xte_sel, num_iteration=model_before.best_iteration_)
    proba_after  = model_after.predict_proba(Xte_sel, num_iteration=model_after.best_iteration_)
    results = {}
    for i, cls_name in enumerate(label_names):
        mask = y_test == i
        conf_b = proba_before[mask, i]
        conf_a = proba_after[mask, i]
        margin_b = (np.sort(proba_before[mask], axis=1)[:, -1] - np.sort(proba_before[mask], axis=1)[:, -2])
        margin_a = (np.sort(proba_after[mask],  axis=1)[:, -1] - np.sort(proba_after[mask],  axis=1)[:, -2])
        entropy_b = -(proba_before[mask] * np.log(proba_before[mask] + 1e-9)).sum(axis=1)
        entropy_a = -(proba_after[mask]  * np.log(proba_after[mask]  + 1e-9)).sum(axis=1)

        results[cls_name] = {
            "confidence_before" : round(float(conf_b.mean()), 4),
            "confidence_after"  : round(float(conf_a.mean()), 4),
            "confidence_delta"  : round(float(conf_a.mean() - conf_b.mean()), 4),
            "margin_before"     : round(float(margin_b.mean()), 4),
            "margin_after"      : round(float(margin_a.mean()), 4),
            "margin_delta"      : round(float(margin_a.mean() - margin_b.mean()), 4),
            "entropy_before"    : round(float(entropy_b.mean()), 4),
            "entropy_after"     : round(float(entropy_a.mean()), 4),
            "entropy_delta"     : round(float(entropy_a.mean() - entropy_b.mean()), 4),
        }
    return results

def decision_boundary_analysis_3way(model_orig, model_smote, model_proposed, Xte_sel, y_test, label_names):
    """Bandingkan Decision Boundary 3 Model (Original vs SMOTE vs ASTRA/RADIANT)"""
    proba_orig  = model_orig.predict_proba(Xte_sel, num_iteration=model_orig.best_iteration_)
    proba_smote = model_smote.predict_proba(Xte_sel, num_iteration=model_smote.best_iteration_)
    proba_prop  = model_proposed.predict_proba(Xte_sel, num_iteration=model_proposed.best_iteration_)

    results = {}
    for i, cls_name in enumerate(label_names):
        mask = y_test == i
        conf_o = proba_orig[mask, i]
        conf_s = proba_smote[mask, i]
        conf_p = proba_prop[mask, i]

        def get_margin(p):
            srt = np.sort(p, axis=1)
            return srt[:, -1] - srt[:, -2]

        margin_o = get_margin(proba_orig[mask])
        margin_s = get_margin(proba_smote[mask])
        margin_p = get_margin(proba_prop[mask])

        def get_entropy(p):
            return -(p * np.log(p + 1e-9)).sum(axis=1)

        entropy_o = get_entropy(proba_orig[mask])
        entropy_s = get_entropy(proba_smote[mask])
        entropy_p = get_entropy(proba_prop[mask])

        results[cls_name] = {
            "conf_orig"    : round(float(conf_o.mean()), 4),
            "conf_smote"   : round(float(conf_s.mean()), 4),
            "conf_proposed": round(float(conf_p.mean()), 4),
            "conf_delta"   : round(float(conf_p.mean() - conf_o.mean()), 4),
            
            "margin_orig"    : round(float(margin_o.mean()), 4),
            "margin_smote"   : round(float(margin_s.mean()), 4),
            "margin_proposed": round(float(margin_p.mean()), 4),
            "margin_delta"   : round(float(margin_p.mean() - margin_o.mean()), 4),

            "entropy_orig"    : round(float(entropy_o.mean()), 4),
            "entropy_smote"   : round(float(entropy_s.mean()), 4),
            "entropy_proposed": round(float(entropy_p.mean()), 4),
            "entropy_delta"   : round(float(entropy_p.mean() - entropy_o.mean()), 4),
        }
    return results

# ==============================================================
# 4. CLUSTER DENSITY & METRICS MODULE
# ==============================================================
def get_2d_embedding(X, y, n_sample=3000, random_state=42):
    """Reduce ke 2D via SVD → t-SNE."""
    if issparse(X): X = X.tocsr()
    if X.shape[0] > n_sample:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(X.shape[0], n_sample, replace=False)
        X, y = X[idx], y[idx]
        
    Xn = normalize(X, norm="l2")
    n_comp = min(50, Xn.shape[1] - 1)
    Xr = TruncatedSVD(n_components=n_comp, random_state=random_state).fit_transform(Xn)
    X2d = TSNE(n_components=2, perplexity=40, metric="cosine", random_state=random_state, init="pca", learning_rate="auto").fit_transform(Xr)
    return X2d, np.asarray(y)

def plot_cluster_density(X2d, y, label_names, title, dir="images", name="image"):
    """Plot Scatter dan KDE Contour Density."""
    classes = np.unique(y)
    fig = plt.figure(figsize=(18, 9))
    fig.suptitle(f"Cluster Density — {title}", fontsize=16, fontweight='bold', color=TEXT_DARK, y=1.01)

    import matplotlib.gridspec as gridspec
    gs = gridspec.GridSpec(2, 3, figure=fig, width_ratios=[2.5, 1, 1], hspace=0.45, wspace=0.35)
    
    ax_scatter = fig.add_subplot(gs[:, 0])
    ax_scatter.set_facecolor(CARD_COLOR)
    ax_scatter.grid(color=GRID_COLOR, linewidth=0.5)
    ax_scatter.spines[['top','right']].set_visible(False)

    for i, c_raw in enumerate(classes):
        c = int(c_raw)
        mask = y == c_raw
        ax_scatter.scatter(X2d[mask, 0], X2d[mask, 1], c=PALETTE[i], label=label_names[c], s=8, alpha=0.45, linewidths=0)
        ax_scatter.scatter(X2d[mask, 0].mean(), X2d[mask, 1].mean(), c=PALETTE[i], s=180, marker='*', edgecolors='white', linewidths=0.8, zorder=5)

    ax_scatter.set_title(f"{title} — t-SNE", fontsize=13, fontweight='bold', color=TEXT_DARK)
    ax_scatter.legend(fontsize=9, frameon=False, markerscale=2, loc='best')
    ax_scatter.set_xlabel("Dim 1", fontsize=10, color=TEXT_MID)
    ax_scatter.set_ylabel("Dim 2", fontsize=10, color=TEXT_MID)

    ax_kde_list = [fig.add_subplot(gs[i // 2, 1 + i % 2]) for i in range(len(classes))]

    for i, (c_raw, ax_k) in enumerate(zip(classes, ax_kde_list)):
        ax_k.set_facecolor(CARD_COLOR)
        ax_k.grid(color=GRID_COLOR, linewidth=0.5)
        ax_k.spines[['top','right']].set_visible(False)
        c, mask = int(c_raw), y == c_raw
        pts, color = X2d[mask], PALETTE[i]

        try:
            kde = gaussian_kde(pts.T, bw_method=0.25)
            xmin, xmax = X2d[:, 0].min()-2, X2d[:, 0].max()+2
            ymin, ymax = X2d[:, 1].min()-2, X2d[:, 1].max()+2
            xx, yy = np.mgrid[xmin:xmax:80j, ymin:ymax:80j]
            zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
            ax_k.contourf(xx, yy, zz, levels=12, cmap=LinearSegmentedColormap.from_list("", ["#F8F9FB", color], N=256), alpha=0.85)
            ax_k.contour(xx, yy, zz, levels=6, colors=[color], linewidths=0.6, alpha=0.5)
        except Exception:
            pass

        ax_k.scatter(pts[:, 0], pts[:, 1], c=color, s=5, alpha=0.3, linewidths=0)
        ax_k.set_title(label_names[c], fontsize=11, fontweight='bold', color=color)
        ax_k.set_xlabel("Dim 1", fontsize=9, color=TEXT_MID)
        ax_k.set_ylabel("Dim 2", fontsize=9, color=TEXT_MID)

    fig.tight_layout()
    save_to_dir(dir, name)

def compute_cluster_metrics_full(X, y, label_names, title=""):
    """Hitung semua metrik cluster di feature space asli."""
    if issparse(X): X = X.tocsr()

    if X.shape[0] > 5000:
        rng = np.random.default_rng(42)
        idx = rng.choice(X.shape[0], 5000, replace=False)
        X_s, y_s = X[idx], np.asarray(y)[idx]
    else:
        X_s, y_s = X, np.asarray(y)

    Xn = normalize(X_s, norm="l2")
    classes = np.unique(y_s)
    C = len(classes)

    sil = silhouette_score(Xn, y_s, metric="cosine")

    centroids = []
    for c in classes:
        mu = np.asarray(Xn[y_s == c].mean(axis=0)).ravel()
        mu = mu / (np.linalg.norm(mu) + 1e-9)
        centroids.append(mu)
    centroids = np.vstack(centroids)

    inter_dists = []
    inter_pairs = {}
    for i in range(C):
        for j in range(i+1, C):
            d = 1 - np.dot(centroids[i], centroids[j])
            inter_dists.append(d)
            inter_pairs[f"{label_names[classes[i]]} ↔ {label_names[classes[j]]}"] = round(d, 4)
            
    mean_inter = np.mean(inter_dists)
    min_inter  = np.min(inter_dists)

    intra_per_class = {}
    for i, c in enumerate(classes):
        Xc = normalize(np.asarray(Xn[y_s == c].todense() if issparse(Xn) else Xn[y_s == c]), norm="l2")
        dists = 1 - (Xc @ centroids[i])
        intra_per_class[label_names[c]] = {
            "mean" : round(float(np.mean(dists)), 4),
            "std"  : round(float(np.std(dists)),  4),
            "n"    : int(np.sum(y_s == c)),
        }
    mean_intra = np.mean([v["mean"] for v in intra_per_class.values()])

    X_arr = Xn.toarray() if issparse(Xn) else Xn
    try: db_idx = davies_bouldin_score(X_arr, y_s)
    except Exception: db_idx = float('nan')

    try: ch_idx = calinski_harabasz_score(X_arr, y_s)
    except Exception: ch_idx = float('nan')

    overlap = mean_intra / (mean_intra + mean_inter + 1e-9)

    return {
        "title"             : title,
        "n_samples"         : X_s.shape[0],
        "silhouette"        : round(sil, 4),
        "mean_inter_dist"   : round(mean_inter, 4),
        "min_inter_dist"    : round(min_inter,  4),
        "mean_intra_dist"   : round(mean_intra, 4),
        "overlap_score"     : round(overlap, 4),
        "davies_bouldin"    : round(db_idx, 4),
        "calinski_harabasz" : round(ch_idx, 1),
        "inter_pairs"       : inter_pairs,
        "intra_per_class"   : intra_per_class,
    }

def print_cluster_summary(m):
    """Print ringkasan metrik klaster."""
    print(f"\n{'='*55}")
    print(f"  {m['title']}")
    print(f"{'='*55}")
    print(f"  Samples           : {m['n_samples']:,}")
    print(f"  Silhouette Score  : {m['silhouette']:+.4f}  {'↑ better' if m['silhouette'] > 0 else '↓ overlap'}")
    print(f"  Mean Inter-dist   : {m['mean_inter_dist']:.4f}  (antar centroid, higher=lebih terpisah)")
    print(f"  Min  Inter-dist   : {m['min_inter_dist']:.4f}  (pasangan paling mirip)")
    print(f"  Mean Intra-dist   : {m['mean_intra_dist']:.4f}  (kompaktness, lower=lebih padat)")
    print(f"  Overlap Score     : {m['overlap_score']:.4f}  (lower=lebih terpisah)")
    print(f"  Davies-Bouldin    : {m['davies_bouldin']:.4f}  (lower=better)")
    print(f"  Calinski-Harabasz : {m['calinski_harabasz']:.1f}  (higher=better)")

def plot_cluster_metrics_comparison(mb, ms, m_prop=None, proposed_name="SMOTE Basic", dir="images", name="image"):
    metric_plots = [
        ("silhouette",        "Silhouette Score",          True,  "higher = better"),
        ("mean_inter_dist",   "Mean Inter-class Distance", True,  "higher = more separated"),
        ("mean_intra_dist",   "Mean Intra-class Distance", False, "lower = more compact"),
        ("overlap_score",     "Overlap Score",             False, "lower = less overlap"),
        ("davies_bouldin",    "Davies-Bouldin Index",      False, "lower = better"),
        ("calinski_harabasz", "Calinski-Harabasz Index",   True,  "higher = better"),
    ]

    for key, title, higher_better, note in metric_plots:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.set_facecolor(CARD_COLOR); ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.6); ax.set_axisbelow(True); ax.spines[['top', 'right']].set_visible(False)

        # ── LOGIKA BERCABANG (2 BAR VS 3 BAR) ──
        if m_prop is None:
            # Mode 2-Way (Notebook 2)
            labels = ["Original", proposed_name]
            colors = [PALETTE[0], PALETTE[1]]
            vals   = [mb[key], ms[key]]
            
            delta_val = vals[1] - vals[0]
            delta_text = f"Δ (SMOTE - Ori) = {delta_val:+.4f}"
        else:
            # Mode 3-Way (Notebook 3A / 3B)
            labels = ["Original", "SMOTE Basic", proposed_name]
            colors = [PALETTE[0], PALETTE[1], PALETTE[2]]
            vals   = [mb[key], ms[key], m_prop[key]]
            
            delta_val = vals[2] - vals[1]
            short_name = "ASTRA" if "ASTRA" in proposed_name else "RADIANT"
            delta_text = f"Δ ({short_name} - SMOTE) = {delta_val:+.4f}"

        # Gambar Batang
        bars = ax.bar(labels, vals, color=colors, edgecolor=BG_COLOR, linewidth=1.5, width=0.5)

        # Angka di atas batang
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max(vals) * 0.02),
                    f"{val:.4f}", ha='center', va='bottom', fontsize=11, fontweight='bold', color=TEXT_MID)

        # Anotasi Delta
        is_improved = (delta_val > 0) == higher_better
        color_delta = "#2A8C7E" if is_improved else "#A63D2F"
        symbol = "✓" if is_improved else "!"
        
        ax.text(0.98, 0.95, f"{delta_text}  {symbol}", transform=ax.transAxes, 
                ha='right', va='top', fontsize=11, fontweight='bold', color=color_delta)

        ax.set_title(f"{title}\n({note})")
        ax.set_ylabel(title)

        max_v, min_v = max(vals), min(0, min(vals) * 1.2)
        ax.set_ylim(min_v, max_v * 1.25)

        fig.tight_layout()
        name = f"cluster_metric_{key}_{proposed_name.lower().replace('-', '_')}"
        save_to_dir(dir, name)


def plot_intra_class_compactness(mb, ms, m_prop=None, label_names=None, proposed_name="SMOTE Basic", dir="images", name="image"):
    """
    Visualisasi kepadatan intra-class per dialek.
    Bisa 2-way atau 3-way comparison.
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_facecolor(CARD_COLOR); ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.6); ax.set_axisbelow(True); ax.spines[['top', 'right']].set_visible(False)
    
    classes_list = list(label_names.values()) if isinstance(label_names, dict) else list(label_names)
    x_pos = np.arange(len(classes_list))
    
    vals_b, stds_b = np.array([mb["intra_per_class"][c]["mean"] for c in classes_list]), np.array([mb["intra_per_class"][c]["std"] for c in classes_list])
    vals_s, stds_s = np.array([ms["intra_per_class"][c]["mean"] for c in classes_list]), np.array([ms["intra_per_class"][c]["std"] for c in classes_list])

    # ── LOGIKA BERCABANG (2 BAR VS 3 BAR) ──
    if m_prop is None:
        # Mode 2-Way
        w = 0.35 
        bars_b = ax.bar(x_pos - w/2, vals_b, width=w, color=PALETTE[0], label="Original", 
                        edgecolor=BG_COLOR, yerr=stds_b, capsize=4, error_kw=dict(elinewidth=1, ecolor=TEXT_LIGHT))
        bars_s = ax.bar(x_pos + w/2, vals_s, width=w, color=PALETTE[1], label=proposed_name, 
                        edgecolor=BG_COLOR, yerr=stds_s, capsize=4, error_kw=dict(elinewidth=1, ecolor=TEXT_LIGHT))
        
        all_bars = list(bars_b) + list(bars_s)
        all_vals = np.concatenate([vals_b, vals_s])
        all_stds = np.concatenate([stds_b, stds_s])
        max_y_val = max(np.max(vals_b + stds_b), np.max(vals_s + stds_s))
        ax.set_title(f"Intra-class Compactness per Dialect — Original vs {proposed_name}")

    else:
        # Mode 3-Way
        w = 0.25 
        vals_a, stds_a = np.array([m_prop["intra_per_class"][c]["mean"] for c in classes_list]), np.array([m_prop["intra_per_class"][c]["std"] for c in classes_list])
        
        bars_b = ax.bar(x_pos - w, vals_b, width=w, color=PALETTE[0], label="Original", 
                        edgecolor=BG_COLOR, yerr=stds_b, capsize=4, error_kw=dict(elinewidth=1, ecolor=TEXT_LIGHT))
        bars_s = ax.bar(x_pos, vals_s, width=w, color=PALETTE[1], label="After SMOTE Basic", 
                        edgecolor=BG_COLOR, yerr=stds_s, capsize=4, error_kw=dict(elinewidth=1, ecolor=TEXT_LIGHT))
        bars_a = ax.bar(x_pos + w, vals_a, width=w, color=PALETTE[2], label=f"After {proposed_name}", 
                        edgecolor=BG_COLOR, yerr=stds_a, capsize=4, error_kw=dict(elinewidth=1, ecolor=TEXT_LIGHT))
        
        all_bars = list(bars_b) + list(bars_s) + list(bars_a)
        all_vals = np.concatenate([vals_b, vals_s, vals_a])
        all_stds = np.concatenate([stds_b, stds_s, stds_a])
        max_y_val = max(np.max(vals_b + stds_b), max(np.max(vals_s + stds_s), np.max(vals_a + stds_a)))
        ax.set_title(f"Intra-class Compactness per Dialect — Original vs SMOTE vs {proposed_name}")

    for bar, val, std in zip(all_bars, all_vals, all_stds):
        ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.005, 
                f"{val:.3f}", ha='center', va='bottom', fontsize=8, color=TEXT_MID, fontweight='bold')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(classes_list)
    ax.set_ylabel("Mean Intra-class Distance\n(lower = more compact)")
    ax.legend(frameon=False)
    ax.set_ylim(0, max_y_val * 1.25)

    fig.tight_layout()
    name = f"cluster_intra_comparison_{proposed_name.lower().replace('-', '_')}"
    save_to_dir(dir, name)


Overwriting utils_ml.py
